# residual-skip-add composite — cx25: residual block as an nn.Module subclass

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `residual-skip-add`, `nn-module-subclass`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import torch.nn.functional as F

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "residual-skip-add"
DD_ATOM_IDS = ["residual-skip-add", "nn-module-subclass"]
DD_SUBTOPICS = ["CNN: Residual skip-connection add", "PyTorch: nn.Module subclassing"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

A ResNet **residual block** has two ingredients:
1. A 'main path' — usually `conv -> bn -> relu -> conv -> bn`.
2. A skip connection that ADDS the input back to the main-path output BEFORE the final ReLU: `out = relu(main(x) + x)`.

The canonical way to package this in PyTorch is to **subclass `nn.Module`**. The subclass registers its child modules in `__init__` (so `.parameters()` finds their weights, and `.to(device)` moves them) and implements the residual-add in `forward`.

**Anatomy.**
```python
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()                  # MUST call — wires up the registry.
        self.conv1 = nn.Conv2d(...)
        self.conv2 = nn.Conv2d(...)
        # ... bn layers ...

    def forward(self, x):
        identity = x                         # save the skip.
        out = self.conv2(F.relu(self.conv1(x)))
        return F.relu(out + identity)        # residual-skip-add HERE.
```

**Why both atoms together.** The skip-add is meaningless without the module to host it. The subclass is the smallest unit that PyTorch's optimizer, `.train()/.eval()`, and `.to(device)` all hook into.

### Composite Exercise — residual block as an nn.Module subclass

**Atoms exercised together**: `residual-skip-add`, `nn-module-subclass`

Implement a class `ResidualBlock(nn.Module)` so that for a square `(N, C, H, W)` input:

- `__init__(self, channels)` calls `super().__init__()` and registers two convs:
  - `self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False)`
  - `self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False)`
- `forward(self, x)` computes `h = F.relu(self.conv1(x))`, then `out = self.conv2(h)`, then **adds the skip `x`**, then applies a final `F.relu`.

Return the constructed class (not an instance) from `cx25_make_residual_block()`.

The test instantiates it, checks `super().__init__()` was called (otherwise `list(block.parameters())` would be empty), and checks the residual-add by ZEROING the conv weights — with zero convs the output must equal `relu(x)`.

In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

def cx25_make_residual_block():
    """Return the ResidualBlock class."""
    raise NotImplementedError

def _test_cx25():
    ResidualBlock = cx25_make_residual_block()
    assert isinstance(ResidualBlock, type) and issubclass(ResidualBlock, nn.Module), (
        'cx25 must return a class that subclasses nn.Module'
    )

    # Case A: instantiation & parameter registration (proves super().__init__() was called).
    t.manual_seed(0)
    block = ResidualBlock(channels=4)
    params = list(block.parameters())
    assert len(params) >= 2, (
        f'block has only {len(params)} parameters — did you forget super().__init__()?'
    )
    # conv1 and conv2 must be registered as submodules.
    named = dict(block.named_children())
    assert 'conv1' in named and 'conv2' in named, f'expected conv1, conv2 children; got {list(named)}'
    assert isinstance(named['conv1'], nn.Conv2d) and isinstance(named['conv2'], nn.Conv2d)

    # Case B: shape contract — (N, C, H, W) in, (N, C, H, W) out (padding=1 + k=3 preserves H,W).
    x = t.randn(2, 4, 5, 7)
    out = block(x)
    assert tuple(out.shape) == (2, 4, 5, 7), f'expected (2,4,5,7), got {tuple(out.shape)}'

    # Case C: residual-skip-add is the LOAD-BEARING op.
    # Zero the conv weights — main path becomes 0, so out = relu(0 + x) = relu(x).
    with t.no_grad():
        block.conv1.weight.zero_()
        block.conv2.weight.zero_()
    out = block(x)
    expected = F.relu(x)  # If skip-add is missing, this would be ~0.
    assert t.allclose(out, expected, atol=1e-6), (
        'with zeroed convs out should equal relu(x) — did you add the skip?'
    )
    # Sanity: out is NOT all-zero (would happen if the skip were missing).
    assert out.abs().sum().item() > 0, 'out is all-zero — skip connection missing'

    # Case D: residual-add is BEFORE the final relu (not after).
    # Build x with negative entries that, after adding to zero main-path, still get relu'd.
    x_neg = -t.ones(1, 4, 3, 3)
    out = block(x_neg)
    # relu(0 + (-1)) = 0 — every entry should be 0.
    assert t.allclose(out, t.zeros_like(out)), 'final relu should clip negatives — got nonzero'
    _dd_passed.add('cx25')

_test_cx25()

<details><summary>Show solution — cx25</summary>

```python
def cx25_make_residual_block():
    class ResidualBlock(nn.Module):
        def __init__(self, channels):
            # Atom B (nn-module-subclass): super().__init__() wires up the module registry.
            super().__init__()
            self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False)
            self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False)

        def forward(self, x):
            identity = x
            h = F.relu(self.conv1(x))
            out = self.conv2(h)
            # Atom A (residual-skip-add): add the skip BEFORE the final activation.
            return F.relu(out + identity)

    return ResidualBlock
```

The `super().__init__()` call is non-negotiable — without it the `nn.Module` base never initialises its parameter/module dicts, and `block.parameters()` returns nothing. The `out + identity` is the canonical residual-add: same shape on both sides because conv with `padding=1, kernel=3` is shape-preserving. The final `F.relu` AFTER the add is the ResNet-v1 ordering — v2 moves the activation before the add (pre-activation), but v1 is the ARENA default.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx25'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx25',
        'subtopics': ["CNN: Residual skip-connection add", "PyTorch: nn.Module subclassing"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()